In [ ]:
"""
dataloader.py
─────────────
Builds train_loader and val_loader for SEED / SEED-IV EEG datasets.

Two workflows are supported:

  A) Cross-subject  – hold out one subject entirely for validation.
                      Best for showing the contrastive loss generalises
                      across people. Recommended for the class project.

  B) Within-subject – split each subject's trials into train / val.
                      Easier, higher accuracy numbers.

SEED-IV folder layout assumed
──────────────────────────────
  seed_iv/
    eeg_raw_data/          ← .mat files, one per (subject, session)
      1/
        1_20160518.mat
        2_20160518.mat
        3_20160518.mat
      2/
        ...
      15/
        ...
    Preprocessed_EEG/      ← alternative: pre-extracted DE features
      ...
    Session1_labels.mat    ← emotion labels per trial per session (0-3)
    Session2_labels.mat
    Session3_labels.mat

If you have DE features instead of raw signals, set USE_RAW=False in
build_loaders() and point data_root at the Preprocessed_EEG folder.

Quick-start
───────────
    from dataloader import build_loaders

    train_loader, val_loader = build_loaders(
        data_root   = '/path/to/seed_iv',
        val_subject = 1,           # hold this subject out for validation
        dataset     = 'SEED-IV',   # or 'SEED'
        window_sec  = 1.0,
        sfreq       = 200,
        batch_size  = 32,
        n_per_class = 8,
    )
"""

import os
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from model import EEGDataset, BalancedBatchSampler

# ─────────────────────────────────────────────────────────────────────────────
# SEED / SEED-IV emotion label maps
# ─────────────────────────────────────────────────────────────────────────────

# SEED-IV: 0=neutral 1=sad 2=fear 3=happy  (3 sessions × 24 trials each)
SEED_IV_LABELS = {
    1: [1,2,3,0,2,0,0,1,0,1,2,1,1,1,2,3,2,2,3,3,0,3,0,3],
    2: [2,1,3,0,0,2,0,2,3,3,2,3,2,0,1,1,2,1,0,3,0,1,3,1],
    3: [1,2,2,1,3,3,3,1,1,2,1,0,2,3,3,0,2,3,0,0,2,0,1,0],
}

# SEED: 0=negative 1=neutral 2=positive  (3 sessions × 15 trials each)
SEED_LABELS = {
    1: [1,0,2,0,1,1,2,0,1,2,2,1,0,2,0],
    2: [2,1,0,0,2,1,1,2,0,2,1,2,0,1,0],
    3: [1,2,0,1,2,0,1,2,0,1,2,0,1,2,0],
}


# ─────────────────────────────────────────────────────────────────────────────
# 1. LOW-LEVEL:  .mat file loading
# ─────────────────────────────────────────────────────────────────────────────

def _load_mat(path: str) -> dict:
    """Load a .mat file, supporting both old (<7.3) and HDF5 (≥7.3) formats."""
    try:
        import scipy.io as sio
        return sio.loadmat(path)
    except Exception:
        # HDF5 / MATLAB v7.3
        try:
            import h5py
            with h5py.File(path, 'r') as f:
                return {k: np.array(v) for k, v in f.items() if not k.startswith('#')}
        except ImportError:
            raise ImportError(
                "h5py is required for MATLAB v7.3 files. "
                "Install with: pip install h5py"
            )


# ─────────────────────────────────────────────────────────────────────────────
# 2. SEGMENTATION helper
# ─────────────────────────────────────────────────────────────────────────────

def _segment_trial(
    eeg   : np.ndarray,   # (Chans, total_samples)  raw EEG for one trial
    label : int,
    sfreq : int,
    window_sec  : float,
    step_sec    : float,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Sliding-window segmentation of a single trial.

    Returns
    -------
    X : (n_windows, Chans, window_samples)
    y : (n_windows,)
    """
    win   = int(window_sec * sfreq)
    step  = int(step_sec   * sfreq)
    T     = eeg.shape[1]
    X, y  = [], []

    for start in range(0, T - win + 1, step):
        X.append(eeg[:, start: start + win])
        y.append(label)

    if not X:
        return np.empty((0, eeg.shape[0], win)), np.empty((0,), dtype=np.int64)

    return np.stack(X).astype(np.float32), np.array(y, dtype=np.int64)


# ─────────────────────────────────────────────────────────────────────────────
# 3. PER-SUBJECT loading
# ─────────────────────────────────────────────────────────────────────────────

def load_subject_raw(
    data_root  : str,
    subject_id : int,           # 1-indexed (1 … 15)
    dataset    : str = 'SEED-IV',
    sfreq      : int = 200,
    window_sec : float = 1.0,
    step_sec   : float = 0.5,   # 50 % overlap
    sessions   : list = None,   # None → all 3 sessions
) -> tuple[np.ndarray, np.ndarray]:
    """
    Load and segment all trials for one subject across (optionally chosen) sessions.

    Returns
    -------
    X    : (N, Chans, window_samples)  float32
    y    : (N,)                        int64
    subj : (N,)                        int64   filled with subject_id-1
    """
    label_map = SEED_IV_LABELS if dataset == 'SEED-IV' else SEED_LABELS
    sessions  = sessions or [1, 2, 3]

    all_X, all_y = [], []

    for sess in sessions:
        sess_dir = os.path.join(data_root, 'eeg_raw_data', str(subject_id))

        # find the .mat file for this session
        # naming convention: {sess}_{date}.mat
        candidates = [
            f for f in os.listdir(sess_dir)
            if f.startswith(f'{sess}_') and f.endswith('.mat')
        ]
        if not candidates:
            print(f"  [warn] no file for subject {subject_id} session {sess} – skipping")
            continue

        mat_path  = os.path.join(sess_dir, candidates[0])
        mat_data  = _load_mat(mat_path)
        labels    = label_map[sess]

        # keys that hold EEG arrays – skip metadata keys
        eeg_keys = sorted([
            k for k in mat_data
            if not k.startswith('__') and hasattr(mat_data[k], 'shape')
               and mat_data[k].ndim == 2
        ])

        for trial_idx, key in enumerate(eeg_keys):
            if trial_idx >= len(labels):
                break                           # safety: skip extra keys
            eeg   = mat_data[key]               # (Chans, samples)
            label = labels[trial_idx]

            Xs, ys = _segment_trial(eeg, label, sfreq, window_sec, step_sec)
            all_X.append(Xs)
            all_y.append(ys)

    if not all_X:
        raise RuntimeError(
            f"No data loaded for subject {subject_id}. "
            "Check data_root and folder layout."
        )

    X = np.concatenate(all_X, axis=0)
    y = np.concatenate(all_y, axis=0)
    return X, y


# ─────────────────────────────────────────────────────────────────────────────
# 4. Z-SCORE NORMALISATION  (per-channel, fit on train, applied to val)
# ─────────────────────────────────────────────────────────────────────────────

class ChannelNormalizer:
    """
    Z-score each EEG channel independently.
    Statistics are computed on the training set and reused for validation/test.

    Usage
    -----
        norm = ChannelNormalizer()
        X_train = norm.fit_transform(X_train)
        X_val   = norm.transform(X_val)
    """
    def __init__(self):
        self.mean_ = None   # (1, Chans, 1)
        self.std_  = None

    def fit(self, X: np.ndarray):
        """X : (N, Chans, Samples)"""
        self.mean_ = X.mean(axis=(0, 2), keepdims=True)     # (1, C, 1)
        self.std_  = X.std (axis=(0, 2), keepdims=True).clip(min=1e-8)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        return (X - self.mean_) / self.std_

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        return self.fit(X).transform(X)


# ─────────────────────────────────────────────────────────────────────────────
# 5. OPTIONAL AUGMENTATIONS
# ─────────────────────────────────────────────────────────────────────────────

class GaussianNoise:
    """Add small Gaussian noise to a (1, Chans, Samples) tensor."""
    def __init__(self, std=0.05):
        self.std = std
    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        return x + torch.randn_like(x) * self.std


class TemporalShift:
    """Randomly roll the time axis by up to `max_shift` samples."""
    def __init__(self, max_shift=10):
        self.max_shift = max_shift
    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        shift = torch.randint(-self.max_shift, self.max_shift + 1, (1,)).item()
        return torch.roll(x, shift, dims=-1)


class ComposeTransforms:
    def __init__(self, transforms):
        self.transforms = transforms
    def __call__(self, x):
        for t in self.transforms:
            x = t(x)
        return x


# ─────────────────────────────────────────────────────────────────────────────
# 6. SPLIT STRATEGIES
# ─────────────────────────────────────────────────────────────────────────────

def cross_subject_split(
    data_root   : str,
    val_subject : int,          # 1-indexed subject to hold out
    dataset     : str = 'SEED-IV',
    n_subjects  : int = 15,
    **load_kwargs,
) -> tuple:
    """
    Load all subjects. Hold out `val_subject` entirely for validation.
    All other subjects form the training set.

    Returns
    -------
    (X_train, y_train, subj_train), (X_val, y_val, subj_val)
    """
    train_X, train_y, train_s = [], [], []
    val_X,   val_y,   val_s   = [], [], []

    for sid in range(1, n_subjects + 1):
        print(f"  Loading subject {sid:02d}/{n_subjects} …", end=' ')
        X, y = load_subject_raw(data_root, sid, dataset=dataset, **load_kwargs)
        s    = np.full(len(y), sid - 1, dtype=np.int64)   # 0-indexed
        print(f"{len(y)} windows")

        if sid == val_subject:
            val_X.append(X);   val_y.append(y);   val_s.append(s)
        else:
            train_X.append(X); train_y.append(y); train_s.append(s)

    return (
        np.concatenate(train_X), np.concatenate(train_y), np.concatenate(train_s),
        np.concatenate(val_X),   np.concatenate(val_y),   np.concatenate(val_s),
    )


def within_subject_split(
    data_root   : str,
    subject_id  : int,
    val_sessions: list = None,   # sessions to use as val; default = [3]
    dataset     : str = 'SEED-IV',
    **load_kwargs,
) -> tuple:
    """
    Single-subject split: train on sessions 1-2, validate on session 3.

    Returns
    -------
    (X_train, y_train, subj_train), (X_val, y_val, subj_val)
    """
    val_sessions   = val_sessions or [3]
    train_sessions = [s for s in [1, 2, 3] if s not in val_sessions]

    X_tr, y_tr = load_subject_raw(data_root, subject_id, dataset=dataset,
                                   sessions=train_sessions, **load_kwargs)
    X_va, y_va = load_subject_raw(data_root, subject_id, dataset=dataset,
                                   sessions=val_sessions, **load_kwargs)

    s_tr = np.full(len(y_tr), subject_id - 1, dtype=np.int64)
    s_va = np.full(len(y_va), subject_id - 1, dtype=np.int64)

    return X_tr, y_tr, s_tr, X_va, y_va, s_va


# ─────────────────────────────────────────────────────────────────────────────
# 7. MAIN FACTORY  →  returns (train_loader, val_loader)
# ─────────────────────────────────────────────────────────────────────────────

def build_loaders(
    data_root     : str,
    val_subject   : int   = 1,      # only used when strategy='cross_subject'
    subject_id    : int   = 1,      # only used when strategy='within_subject'
    strategy      : str   = 'cross_subject',   # 'cross_subject' | 'within_subject'
    dataset       : str   = 'SEED-IV',
    sfreq         : int   = 200,
    window_sec    : float = 1.0,
    step_sec      : float = 0.5,
    n_per_class   : int   = 8,      # samples per class per batch
    num_workers   : int   = 4,
    augment_train : bool  = True,
    val_sessions  : list  = None,
) -> tuple[DataLoader, DataLoader]:
    """
    One-call factory that handles loading, normalisation, and DataLoader creation.

    Parameters
    ----------
    data_root     : path to the SEED / SEED-IV root directory
    val_subject   : (cross-subject) subject index (1–15) held out for val
    subject_id    : (within-subject) which subject to use
    strategy      : 'cross_subject' or 'within_subject'
    dataset       : 'SEED-IV' (4 classes) or 'SEED' (3 classes)
    sfreq         : sampling frequency of the EEG data
    window_sec    : segmentation window length in seconds
    step_sec      : sliding-window step in seconds (< window_sec → overlap)
    n_per_class   : samples per class in each balanced training batch
    num_workers   : DataLoader worker processes
    augment_train : if True, apply Gaussian noise + temporal shift to training windows
    val_sessions  : (within-subject) which sessions to use as validation

    Returns
    -------
    train_loader, val_loader
    """

    nb_classes = 4 if dataset == 'SEED-IV' else 3

    # ── 1. load raw arrays ────────────────────────────────────────────────
    print(f"\nBuilding loaders  [{strategy}]  dataset={dataset}")
    load_kw = dict(sfreq=sfreq, window_sec=window_sec, step_sec=step_sec)

    if strategy == 'cross_subject':
        X_tr, y_tr, s_tr, X_va, y_va, s_va = cross_subject_split(
            data_root, val_subject=val_subject, dataset=dataset, **load_kw
        )
    elif strategy == 'within_subject':
        X_tr, y_tr, s_tr, X_va, y_va, s_va = within_subject_split(
            data_root, subject_id=subject_id, dataset=dataset,
            val_sessions=val_sessions, **load_kw
        )
    else:
        raise ValueError(f"Unknown strategy '{strategy}'. "
                         "Choose 'cross_subject' or 'within_subject'.")

    # ── 2. normalise (fit on train only) ──────────────────────────────────
    norm = ChannelNormalizer()
    X_tr = norm.fit_transform(X_tr)
    X_va = norm.transform(X_va)

    print(f"\n  Train : {X_tr.shape}  labels {np.bincount(y_tr)}")
    print(f"  Val   : {X_va.shape}  labels {np.bincount(y_va)}")

    # ── 3. transforms ─────────────────────────────────────────────────────
    train_transform = (
        ComposeTransforms([GaussianNoise(std=0.05), TemporalShift(max_shift=10)])
        if augment_train else None
    )

    # ── 4. datasets ───────────────────────────────────────────────────────
    train_ds = EEGDataset(X_tr, y_tr, subject_id=s_tr, transform=train_transform)
    val_ds   = EEGDataset(X_va, y_va, subject_id=s_va)

    # ── 5. samplers ───────────────────────────────────────────────────────
    # Training uses BalancedBatchSampler so every batch has ≥2 samples per
    # class, which is required for valid positive pairs in the contrastive loss.
    train_sampler = BalancedBatchSampler(
        y_tr, n_per_class=n_per_class, nb_classes=nb_classes
    )
    batch_size = n_per_class * nb_classes

    # Validation uses a simple sequential loader — no balancing needed.
    train_loader = DataLoader(
        train_ds,
        batch_sampler = train_sampler,
        num_workers   = num_workers,
        pin_memory    = True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size  = batch_size,
        shuffle     = False,
        num_workers = num_workers,
        pin_memory  = True,
        drop_last   = False,
    )

    print(f"\n  Batch size    : {batch_size}  ({n_per_class} per class × {nb_classes} classes)")
    print(f"  Train batches : {len(train_loader)}")
    print(f"  Val   batches : {len(val_loader)}\n")

    return train_loader, val_loader


# ─────────────────────────────────────────────────────────────────────────────
# 8. SMOKE TEST WITH SYNTHETIC DATA (no real files needed)
# ─────────────────────────────────────────────────────────────────────────────

def _make_synthetic_loaders(
    nb_classes  = 4,
    n_subjects  = 15,
    n_per_subj  = 400,
    Chans       = 62,
    Samples     = 200,
    n_per_class = 8,
    num_workers = 0,
):
    """
    Returns (train_loader, val_loader) built from randomly generated data.
    Useful for testing the training loop without real EEG files.
    """
    N      = n_subjects * n_per_subj
    X      = np.random.randn(N, Chans, Samples).astype(np.float32)
    y      = np.tile(np.arange(nb_classes), N // nb_classes + 1)[:N]
    subj   = np.repeat(np.arange(n_subjects), n_per_subj)

    # last subject → val
    val_mask = subj == (n_subjects - 1)
    X_tr, y_tr, s_tr = X[~val_mask], y[~val_mask], subj[~val_mask]
    X_va, y_va, s_va = X[ val_mask], y[ val_mask], subj[ val_mask]

    norm = ChannelNormalizer()
    X_tr = norm.fit_transform(X_tr)
    X_va = norm.transform(X_va)

    train_ds = EEGDataset(X_tr, y_tr, subject_id=s_tr,
                          transform=ComposeTransforms([GaussianNoise(), TemporalShift()]))
    val_ds   = EEGDataset(X_va, y_va, subject_id=s_va)

    sampler  = BalancedBatchSampler(y_tr, n_per_class=n_per_class, nb_classes=nb_classes)

    train_loader = DataLoader(train_ds, batch_sampler=sampler, num_workers=num_workers)
    val_loader   = DataLoader(val_ds, batch_size=n_per_class*nb_classes,
                              shuffle=False, num_workers=num_workers)
    return train_loader, val_loader


# ─────────────────────────────────────────────────────────────────────────────
# 9. RUN  (python dataloader.py)
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':
    print("=== Synthetic dataloader smoke-test ===\n")

    train_loader, val_loader = _make_synthetic_loaders(
        nb_classes=4, n_subjects=15, n_per_subj=200
    )

    print(f"Train batches : {len(train_loader)}")
    print(f"Val   batches : {len(val_loader)}")

    x, y, s = next(iter(train_loader))
    print(f"\nFirst train batch")
    print(f"  x shape  : {tuple(x.shape)}   (B, 1, Chans, Samples)")
    print(f"  y shape  : {tuple(y.shape)}   dtype={y.dtype}")
    print(f"  s shape  : {tuple(s.shape)}   (subject ids)")
    print(f"  x range  : [{x.min():.2f}, {x.max():.2f}]")
    for c in range(4):
        print(f"  class {c} count : {(y == c).sum().item()}")

    x_v, y_v, s_v = next(iter(val_loader))
    print(f"\nFirst val batch")
    print(f"  x shape  : {tuple(x_v.shape)}")
    print(f"  y shape  : {tuple(y_v.shape)}")

    print("\nSmoke-test complete ✓")

In [ ]:
from model import build_model, EEGDataset, BalancedBatchSampler, Trainer
from losses import ClassificationLoss, ContrastiveLoss, ContrastivePrototype

model     = build_model(nb_classes=4, Chans=62, Samples=200)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

trainer = Trainer(
    model, ClassificationLoss(), ContrastivePrototype(num_classes=4),
    optimizer, lambda_con=0.5, warmup_epochs=5, device='cuda'
)
history = trainer.fit(train_loader, val_loader, epochs=50)